In [1]:
from build123d import *
from ocp_vscode import *
from math import sin, cos,tan, pi
set_port(3939)

set_defaults(reset_camera=Camera.CENTER, helper_scale=5)

In [61]:
# place things
wall = 1
length = 50
clip_space = 25.5
angle_ratio = 2.5
with BuildPart() as p:
    Box(length, 100, 120, align=(Align.CENTER, Align.MIN))
    topf = p.faces().sort_by(Axis.Z)[-1]
    bopf = p.faces().sort_by(Axis.Z)[0]
    offset(amount=-wall, openings=[topf, bopf])

    #ss = p.edges().sort_by(Axis.Y).sort_by(Axis.Z)[0]
    #loc = list(ss.location.position)
    
    with BuildSketch(Plane.XY):
        with Locations((0, -clip_space, 0)):
            Rectangle(length, wall, align=(Align.CENTER, Align.MAX))
    extrude(amount=33)

    clip_main_top = p.faces(Select.LAST).sort_by(Axis.Z)[-1]
    clip_main_top_inner = clip_main_top.edges().sort_by(Axis.Y)[-1].center()
    clip_connection_workplane = Plane(
        origin=clip_main_top_inner, x_dir=(1, 0, 0), z_dir=(0, 1, angle_ratio)
    )
    with BuildSketch(clip_connection_workplane.offset(-0.5 * wall)):
        Rectangle(length, wall, align=(Align.CENTER, Align.MAX))
    extrude(amount=angle_ratio*(clip_space + 2*(1.5 * wall)))

    fillet([e for e in p.edges() if e.location.position.Z > 0], radius=0.01)
    overall_top = p.faces().sort_by(Axis.Z)[-1]
    fillet(overall_top.edges(), radius=0.4)

    
show(p)

+


In [62]:
export_stl(p.part, "kitchen_draw_divider.stl")

True